<a href="https://colab.research.google.com/github/Kavia-M/Data_science_course-Intellipaat/blob/main/Python/Classwork/Trainers_notes/Gen-AI/EN_DE_28_11.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#ENGLISH - FRENCH

In [ ]:
#encoder - english - token - lstm- h,c
#decoder - french - token -lstm -output



In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input,Embedding,LSTM,Dense

In [ ]:
#step-2

eng = ["hi","hello","how are you","thank you"]
fr = ["salut", "bonjour","comment ca va","merci"]

#TO do token
- tokenizer()

- fit_on_texts - read all the sentences and find the unique words - number id

- text to sequences -convert sentences into list of ID's

- pad_sequences- add zeros to make sentences(interger id's) - iut should be in same length



In [ ]:
#step-3
tok_eng = Tokenizer()
tok_eng.fit_on_texts(eng)
x = tok_eng.texts_to_sequences(eng)
x = pad_sequences(x)


In [ ]:
x

array([[0, 0, 2],
       [0, 0, 3],
       [4, 5, 1],
       [0, 6, 1]], dtype=int32)

In [ ]:
#step-4 french
tok_fr = Tokenizer()
tok_fr.fit_on_texts(fr)
y = tok_fr.texts_to_sequences(fr)
y = pad_sequences(y)

In [ ]:
y

array([[0, 0, 1],
       [0, 0, 2],
       [3, 4, 5],
       [0, 0, 6]], dtype=int32)

In [ ]:
y_in = y[:,:-1] #input to decoder
y_out = y[:,1:] #target output

In [ ]:
y_in

array([[0, 0],
       [0, 0],
       [3, 4],
       [0, 0]], dtype=int32)

In [ ]:
y_out

array([[0, 1],
       [0, 2],
       [4, 5],
       [0, 6]], dtype=int32)

#IN neural networks - seq data - 3d

- how mnay sentences are we processing at once? --> batch size

- how many words(timesteps) are in each sentence

- what is the representation size of each word?

- input - 3d

- (batch_size, time_steps,features(each word contain - numbers))

In [ ]:
y_out = y_out.reshape((y_out.shape[0], y_out.shape[1],1))

In [ ]:
y_out

array([[[0],
        [1]],

       [[0],
        [2]],

       [[4],
        [5]],

       [[0],
        [6]]], dtype=int32)

In [ ]:
vocab_eng = len(tok_eng.word_index) +1
vocab_fr = len(tok_fr.word_index)+1

return_sequence = True/False
Do we want our LSTM output for every timestep or justat the last step??

- return_sequences - false - for encoder - by default

- return_sequences - True - for decoder



#return_state

- should LSTM also return its internal memory - h,c

- return_state - False - by default - just return your output - no final h,c


- True - output, final h,c









In [ ]:
#build the encoder

enc_in  = Input(shape = (x.shape[1],))
enc_emb = Embedding(vocab_eng,8)(enc_in)
_,h,c = LSTM(32,return_state= True)(enc_emb)


In [ ]:
#build your decoder

dec_in = Input(shape = (y_in.shape[1],))
dec_emb = Embedding(vocab_fr,8)(dec_in)
dec_out,_,_ = LSTM(32, return_sequences= True, return_state= True)(dec_emb , initial_state = [h,c])
out = Dense(vocab_fr,activation = 'softmax')(dec_out)


In [ ]:
dec_emb

<KerasTensor shape=(None, 2, 8), dtype=float32, sparse=False, ragged=False, name=keras_tensor_29>

In [ ]:
model = Model([enc_in,dec_in],out)
model.compile(optimizer ='adam',loss = 'sparse_categorical_crossentropy')

In [ ]:
model.fit([x,y_in],y_out,epochs = 300,verbose = 0)

In [ ]:
preds = model.predict([x,y_in])

for i , pred in enumerate(preds):
  ids = np.argmax(pred, axis = 1)
  words = [tok_fr.index_word.get(idx, '???????') for idx in ids]
  print(f"engish : {eng[i]} ------------>>>>> predicted french : {''.join(words)}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 268ms/step
engish : hi ------------>>>>> predicted french : ???????salut
engish : hello ------------>>>>> predicted french : ???????bonjour
engish : how are you ------------>>>>> predicted french : cava
engish : thank you ------------>>>>> predicted french : ???????merci
